In [ ]:
import torch

In [ ]:
step = 1
sdm = torch.load(f'/home/user/projects/my-nanochat/dump_step_{step:05d}_rank_0.pt', map_location='cpu')
sdk = torch.load(f'/home/user/.cache/nanochat/nanochat_step_{step:05d}_rank_0.pt', map_location='cpu')

# sdm = torch.load(f'/home/user/projects/my-nanochat/muon_muonm/dump_step_{step:05d}_rank_0.pt', map_location='cpu')
# sdk = torch.load(f'/home/user/projects/my-nanochat/muon_muonk/dump_step_{step:05d}_rank_0.pt', map_location='cpu')

In [ ]:
if sdm.keys() != sdk.keys():
    print("Keys differ:")
    print("In sdm but not in sdk:", sdm.keys() - sdk.keys())
    print("In sdk but not in sdm:", sdk.keys() - sdm.keys())
else:
    print("Keys are the same.")

In [ ]:
if sdm['step'] != sdk['step']:
    print(f"Step values differ: sdm['step']={sdm['step']}, sdk['step']={sdk['step']}")
else:
    print(f"Step values are the same and equal to {sdm['step']}.")

In [ ]:
if len(sdm['weights_before']) != len(sdk['weights_before']):
    print(f"Lengths of 'weights_before' differ: {len(sdm['weights_before'])} vs {len(sdk['weights_before'])}")
else:
    print(f"Lengths of 'weights_before' are the same and equal to {len(sdm['weights_before'])}.")

In [ ]:
for i, (nwm, nwk) in enumerate(zip(sdm['weights_before'], sdk['weights_before'])):
    name_m, wm = nwm
    name_k, wk = nwk
    if wm.shape != wk.shape:
        print(f'weights_before at index {i} have different shapes: {wm.shape} vs {wk.shape}')
        break
    if wm.dtype != wk.dtype:
        print(f'weights_before at index {i} have different dtypes: {wm.dtype} vs {wk.dtype}')
        break
    print(f"{i}: name={name_m} shape={wm.shape} dtype={wm.dtype} max_diff={(wm - wk).abs().max()}")
else:
    print('All weights_before are equal')

In [ ]:
for i, (nbm, nbk) in enumerate(zip(sdm['buffs_before'].items(), sdk['buffs_before'].items())):
    name_m, bm = nbm
    name_k, bk = nbk
    assert name_m == name_k, f'Buffer names at index {i} differ: {name_m} vs {name_k}'
    if bm.shape != bk.shape:
        print(f'buffs_before at index {i} have different shapes: {bm.shape} vs {bk.shape}')
        break
    if bm.dtype != bk.dtype:
        print(f'buffs_before at index {i} have different dtypes: {bm.dtype} vs {bk.dtype}')
        break
    print(f"{i}: name={name_m} shape={bm.shape} dtype={bm.dtype} max_diff={(bm - bk).abs().max()}")
else:
    print('All buffs_before are equal')

In [ ]:
if len(sdm['x']) != len(sdk['x']):
    print(f"Lengths of 'x' differ: {len(sdm['x'])} vs {len(sdk['x'])}")
else:
    print(f"Lengths of 'x' are the same and equal to {len(sdm['x'])}.")

In [ ]:
# Check all x's are equal
for index, (xm, xk) in enumerate(zip(sdm['x'], sdk['x'])):
    if not (xm == xk).all().item():
        print(f"x at index {index} differs.")
else:
    print("All 'x' tensors are equal.")

In [ ]:
# Check all y's are equal
for index, (ym, yk) in enumerate(zip(sdm['y'], sdk['y'])):
    if not (ym == yk).all().item():
        print(f"y at index {index} differs.")
else:
    print("All 'y' tensors are equal.")

In [ ]:
# Check all logits are equal
for i, (lm, lk) in enumerate(zip(sdm['logits'], sdk['logits'])):
    if lm.shape != lk.shape:
        print(f'logits at index {i} have different shapes: {lm.shape} vs {lk.shape}')
        break
    if lm.dtype != lk.dtype:
        print(f'logits at index {i} have different dtypes: {lm.dtype} vs {lk.dtype}')
        break
    max_diff = (lm - lk).abs().max().item()
    print(f'logits at index {i} max absolute difference: {max_diff}')

In [ ]:
# Check loss_div_accum
if len(sdm['loss_div_accum']) != len(sdk['loss_div_accum']):
    print(f"Lengths of 'loss_div_accum' differ: {len(sdm['loss_div_accum'])} vs {len(sdk['loss_div_accum'])}")
else:
    print(f"Lengths of 'loss_div_accum' are the same and equal to {len(sdm['loss_div_accum'])}.")
for index, (lm, lk) in enumerate(zip(sdm['loss_div_accum'], sdk['loss_div_accum'])):
    loss_diff = abs(lm - lk)
    emo = "🟢" if loss_diff == 0 else "🔴"
    print(f"{emo} loss_div_accum at index {index} with difference {loss_diff}.")

In [ ]:
# Check gradients
for i, (gm, gk) in enumerate(zip(sdm['gradients'], sdk['gradients'])):
    name = sdm['weights_before'][i][0]
    if gm.shape != gk.shape:
        print(f'gradients at index {i} have different shapes: {gm.shape} vs {gk.shape}')
        break
    if gm.dtype != gk.dtype:
        print(f'gradients at index {i} have different dtypes: {gm.dtype} vs {gk.dtype}')
        break
    #print('---')
    print(f"{i}: name={name} shape={gm.shape} dtype={gm.dtype} max_diff={(gm - gk).abs().max()}")
    #print(gm[0, :5])
    #print(gk[0, :5])

In [ ]:
sdm.keys()

In [ ]:
sdm['lrm'], sdk['lrm']

In [ ]:
sdm['muon_momentum'], sdk['muon_momentum']

In [ ]:
def check_optimizer_states(opt_sm, opt_sk):
    assert isinstance(opt_sm, dict)
    assert isinstance(opt_sk, dict)
    assert len(opt_sm) == len(opt_sk)
    assert opt_sm.keys() == opt_sk.keys()
    assert list(opt_sm.keys()) == ['state', 'param_groups']

    # Param Groups
    assert len(opt_sm['param_groups']) == len(opt_sk['param_groups'])
    for j, (pgm, pgk) in enumerate(zip(opt_sm['param_groups'], opt_sk['param_groups'])):
        all_keys = set(pgm.keys()).union(set(pgk.keys()))
        for key in all_keys:
            if key in pgm and key not in pgk:
                if key in ["weight_decay", "ns_coefficients", "eps", "adjust_lr_fn"]:
                    print(f"🔵 Expected missing key '{key}' in opt_sk")
                else:
                    print(f"🟠 Key '{key}' present in opt_sm but missing in opt_sk")
            elif key not in pgm and key in pgk:
                print(f"🟠 Key '{key}' present in opt_sk but missing in opt_sm")
            else:  # key in pgk and key in pgm
                emo = "🟢" if pgm[key] == pgk[key] else "🔴"
                print(f"{emo} Checking optimizer param_groups at index {i}, group {j}, key '{key}'", pgm[key], pgk[key])

    # State
    assert isinstance(opt_sm['state'], dict)
    assert isinstance(opt_sk['state'], dict)
    assert len(opt_sm['state']) == len(opt_sk['state'])
    assert opt_sm['state'].keys() == opt_sk['state'].keys()
    print("keys", opt_sm['state'].keys())
    for j, key in enumerate(opt_sm['state'].keys()):
        print(f"Checking optimizer state for param {j} (key={key})")
        sm = opt_sm['state'][key]
        sk = opt_sk['state'][key]
        assert sm.keys() == sk.keys()
        for state_key in sm.keys():
            print(f"  Checking state key '{state_key}'")
            state_val_m = sm[state_key]
            state_val_k = sk[state_key]
            assert isinstance(state_val_m, torch.Tensor)
            assert type(state_val_m) == type(state_val_k)
            assert state_val_m.shape == state_val_k.shape
            assert state_val_m.dtype == state_val_k.dtype
            max_diff = (state_val_m - state_val_k).abs().max().item()
            emo = "🟢" if max_diff == 0 else "🔴"
            print(f"    {emo} {state_val_m.shape=} {state_val_m.dtype=} max_diff={max_diff}")

In [ ]:
# Check optimizer states
adam_sm = sdm['optimizer_states_before'][0]
adam_sk = sdk['optimizer_states_before'][0]
check_optimizer_states(adam_sm, adam_sk)

In [ ]:
# Check optimizer states before
muon_sm = sdm['optimizer_states_before'][1]
muon_sk = sdk['optimizer_states_before'][1]
check_optimizer_states(muon_sm, muon_sk)

In [ ]:
# Check optimizer states after
adam_sm = sdm['optimizer_states_after'][0]
adam_sk = sdk['optimizer_states_after'][0]
check_optimizer_states(adam_sm, adam_sk)

In [ ]:
# Check optimizer states
muon_sm = sdm['optimizer_states_after'][1]
muon_sk = sdk['optimizer_states_after'][1]
check_optimizer_states(muon_sm, muon_sk)

In [ ]:
# Check weights after
for i, (nwm, nwk) in enumerate(zip(sdm['weights_after'], sdk['weights_after'])):
    name, wm = nwm
    name_k, wk = nwk
    if wm.shape != wk.shape:
        print(f'weights_after at index {i} have different shapes: {wm.shape} vs {wk.shape}')
        break
    if wm.dtype != wk.dtype:
        print(f'weights_after at index {i} have different dtypes: {wm.dtype} vs {wk.dtype}')
        break
    #print('---')
    print(f"{i}: name={name} shape={wm.shape} dtype={wm.dtype} max_diff={(wm - wk).abs().max()}")
    #print(wm[0, :5])
    #print(wk[0, :5])
else:
    print('All weights_after are equal')